# Transformer guideline heads on Kaggle

Trains ModernBERT with 13 guideline heads from `src/heads/` in the team repo, on a Kaggle GPU.
The repo already holds the data, so this notebook only clones it, installs what Kaggle lacks, runs a 20-step trial and then the full runs.

Settings: accelerator GPU T4 x2, internet on. Run it with Save Version, then Save and Run All, so it keeps going after the tab closes.
With two GPUs the runs in `RUNS` go in parallel, one per GPU. With one GPU they go one after another.

The outputs land in `/kaggle/working/runs/<run-name>/`. Download `logits_*.npy`, `cases_*.csv`, `config.json` and the logs into `runs/<run-name>/` on the laptop and run `python -m src.heads.evaluate --run runs/<run-name>` there.

In [ ]:
# Parameters
REPO = "https://github.com/Break-Through-Tech/A3-Consulting-1A-adjudicative-guidelines-analyzer.git"
BRANCH = "transformer-heads"
MODEL = "answerdotai/ModernBERT-base"
# Each run is (run name, max length, extra arguments for src.heads.train).
RUNS = [
    ("modernbert-2048", 2048, []),
    ("modernbert-4096", 4096, []),
]
TRIAL_STEPS = 20
RUN_TRIAL = True
RUN_FULL = True
REPO_DIR = "/kaggle/temp/repo"
OUTPUT_DIR = "/kaggle/working/runs"

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
import os, subprocess
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], check=True)

## 3. Install what Kaggle lacks

Kaggle's own CUDA torch is kept. The repo's pins are for Python 3.14 and Kaggle runs an older Python.

In [ ]:
import importlib.metadata as md
import subprocess, sys

def version_tuple(text):
    return tuple(int(part) for part in text.split("+")[0].split(".")[:3] if part.isdigit())

def installed(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

needs = ["transformers==5.22.2"]
for name, minimum in [("pandas", "2.2.0"), ("pyarrow", "15.0.0"), ("scikit-learn", "1.4.0")]:
    have = installed(name)
    if have is None or version_tuple(have) < version_tuple(minimum):
        needs.append(f"{name}>={minimum}")
if installed("transformers") != "5.22.2" or len(needs) > 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needs], check=True)

## 4. Versions, and the model download

In [ ]:
import sys, torch, transformers, pandas, numpy, pyarrow, sklearn
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
print("transformers", transformers.__version__, "pandas", pandas.__version__, "numpy", numpy.__version__,
      "pyarrow", pyarrow.__version__, "sklearn", sklearn.__version__)
from huggingface_hub import snapshot_download
snapshot_download(MODEL)

## 5. Launcher

Each run is its own process, pinned to one GPU, with its log streamed here.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

import torch

def run_jobs(jobs, poll_seconds=30):
    """jobs: list of (name, argv for src.heads.train). One job per GPU at a time."""
    n_gpus = max(1, torch.cuda.device_count())
    log_dir = Path(OUTPUT_DIR) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    pending = list(jobs)
    active = {}
    failed = []

    def drain(entry, final=False):
        text = entry["log"].read_text(encoding="utf-8", errors="replace")
        chunk = text[entry["offset"]:]
        cut = len(chunk) if final else chunk.rfind("\n") + 1
        for line in chunk[:cut].splitlines():
            line = line.split("\r")[-1]
            if line.strip():
                print(f"[{entry['name']}] {line}", flush=True)
        entry["offset"] += cut

    while pending or active:
        for gpu in range(n_gpus):
            if gpu not in active and pending:
                name, argv = pending.pop(0)
                log = log_dir / f"{name}.log"
                handle = open(log, "w", encoding="utf-8")
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
                           TOKENIZERS_PARALLELISM="false")
                process = subprocess.Popen(
                    [sys.executable, "-m", "src.heads.train", *argv],
                    cwd=REPO_DIR, env=env, stdout=handle, stderr=subprocess.STDOUT,
                )
                active[gpu] = {"name": name, "process": process, "log": log, "offset": 0,
                               "handle": handle, "started": time.time()}
                print(f"[{name}] started on GPU {gpu}", flush=True)
        time.sleep(poll_seconds)
        for gpu, entry in list(active.items()):
            drain(entry)
            code = entry["process"].poll()
            if code is not None:
                entry["handle"].close()
                drain(entry, final=True)
                minutes = (time.time() - entry["started"]) / 60
                print(f"[{entry['name']}] finished with exit code {code} after {minutes:.1f} min", flush=True)
                if code != 0:
                    failed.append(entry["name"])
                del active[gpu]
    if failed:
        raise RuntimeError(f"Runs failed: {failed}. See {log_dir}")

def train_args(name, max_length, extra):
    return ["--model", MODEL, "--max-length", str(max_length), "--run-name", name,
            "--output-dir", OUTPUT_DIR, *extra]

## 6. Trial: 20 steps on 200 cases per run

In [ ]:
if RUN_TRIAL:
    trials = [
        (f"trial-{name}", train_args(f"trial-{name}", max_length, extra)
         + ["--limit", "200", "--max-steps", str(TRIAL_STEPS), "--log-every", "5", "--overwrite"])
        for name, max_length, extra in RUNS
    ]
    run_jobs(trials, poll_seconds=15)
    for name, _ in trials:
        best = Path(OUTPUT_DIR) / name / "best_model.pt"
        if best.exists():
            best.unlink()  # keeps the trial outputs small

## 7. Full runs

In [ ]:
if RUN_FULL:
    run_jobs([(name, train_args(name, max_length, extra)) for name, max_length, extra in RUNS], poll_seconds=60)

## 8. Output files

In [ ]:
for path in sorted(Path(OUTPUT_DIR).rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 2**20:10.1f} MB  {path}")